# Feature Engineering for Classification
## Module 2, Session 5: Creating Better Features to Improve Models

**Goal:** Learn feature engineering techniques to boost model performance  
**Dataset:** Synthetic dataset with engineerable features  
**Problem:** Binary classification with feature engineering  
**Target:** Show 10%+ accuracy improvement through feature engineering  

---

### What You'll Learn
1. What is feature engineering and why it matters
2. Creating interaction features
3. Polynomial features for non-linear relationships
4. Feature scaling (StandardScaler, MinMaxScaler)
5. Binning continuous variables
6. Domain-specific feature creation
7. Before/after accuracy comparison

---

**Created:** 2026-01-06  
**Course:** ML for Engineers  
**Module:** 2 - Classification

## Step 1: Import Libraries

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Preprocessing and Feature Engineering
from sklearn.preprocessing import (
    StandardScaler,
    MinMaxScaler,
    PolynomialFeatures
)
from sklearn.model_selection import train_test_split

# Machine Learning
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Metrics
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
np.random.seed(42)

print("✓ All libraries imported successfully!")

## Step 2: Create a Dataset Perfect for Feature Engineering

We'll create a synthetic dataset with features that have:
- Interaction effects (A × B matters)
- Non-linear relationships (A² matters)
- Different scales (one feature 0-1, another 0-1000)

> **AI Assistant Prompt:**  
> "Create a synthetic binary classification dataset where the target depends on interaction features (X1 * X2) and polynomial features (X1²). Include features with different scales to demonstrate the importance of scaling."

In [ ]:
# Create synthetic dataset
np.random.seed(42)
n_samples = 1000

# Create features with different characteristics
data = {
    # Features with different scales
    'age': np.random.randint(18, 80, n_samples),
    'income': np.random.randint(20000, 150000, n_samples),
    'years_experience': np.random.randint(0, 40, n_samples),
    
    # Features that interact
    'hours_studied': np.random.uniform(0, 100, n_samples),
    'practice_tests': np.random.randint(0, 20, n_samples),
    
    # Features with non-linear effects
    'motivation_score': np.random.uniform(1, 10, n_samples),
}

df = pd.DataFrame(data)

# Create target based on complex patterns
# Success depends on:
# 1. Interaction: hours_studied * practice_tests (synergy)
# 2. Non-linear: motivation_score squared matters more at high levels
# 3. Combined effect: (age * income) / 1000000

score = (
    0.3 * (df['hours_studied'] * df['practice_tests'] / 100) +  # Interaction
    0.4 * (df['motivation_score'] ** 2 / 10) +                  # Non-linear
    0.2 * (df['age'] * df['income'] / 5000000) +                # Interaction with scale
    0.1 * df['years_experience'] +
    np.random.normal(0, 2, n_samples)                            # Noise
)

# Create binary target (top 60% succeed)
threshold = np.percentile(score, 40)
df['success'] = (score > threshold).astype(int)

print("✓ Dataset created!")
print(f"\nDataset shape: {df.shape}")
print(f"Success rate: {df['success'].mean()*100:.1f}%")

print("\n💡 This dataset is designed to benefit from feature engineering:")
print("   • Interaction features (hours × tests)")
print("   • Polynomial features (motivation²)")
print("   • Features at different scales need normalization")

In [ ]:
# Display sample data
print("Sample Data:")
display(df.head(10))

# Basic statistics
print("\nFeature Statistics:")
display(df.describe())

print("\n⚠️ Notice: Features have VERY different scales!")
print(f"   • age: {df['age'].min()}-{df['age'].max()}")
print(f"   • income: {df['income'].min()}-{df['income'].max()}")
print(f"   • motivation: {df['motivation_score'].min():.1f}-{df['motivation_score'].max():.1f}")

## Step 3: Baseline Model (No Feature Engineering)

First, let's train a model with RAW features to establish a baseline.

In [ ]:
print("\n" + "="*70)
print("BASELINE MODEL - NO FEATURE ENGINEERING")
print("="*70)

# Separate features and target
X_baseline = df.drop('success', axis=1)
y = df['success']

# Split data
X_train_base, X_test_base, y_train, y_test = train_test_split(
    X_baseline, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"\nTraining samples: {len(X_train_base)}")
print(f"Testing samples: {len(X_test_base)}")
print(f"Features: {list(X_baseline.columns)}")

# Train Logistic Regression
print("\nTraining Logistic Regression...")
baseline_model = LogisticRegression(max_iter=1000, random_state=42)
baseline_model.fit(X_train_base, y_train)

# Evaluate
baseline_predictions = baseline_model.predict(X_test_base)
baseline_accuracy = accuracy_score(y_test, baseline_predictions)

print("\n" + "="*70)
print(f"BASELINE ACCURACY: {baseline_accuracy*100:.2f}%")
print("="*70)

print("\n⚠️ This is our starting point. Let's improve it with feature engineering!")

In [ ]:
# Detailed baseline report
print("Baseline Classification Report:")
print(classification_report(y_test, baseline_predictions, target_names=['Fail', 'Success']))

## Step 4: Feature Engineering Technique #1 - Interaction Features

**Interaction features** capture relationships between variables.

Example: Studying 50 hours AND taking 10 practice tests is better than studying 100 hours with 0 tests.

In [ ]:
print("\n" + "="*70)
print("FEATURE ENGINEERING #1: INTERACTION FEATURES")
print("="*70)

# Create a copy for feature engineering
df_engineered = df.copy()

# Create interaction features
df_engineered['study_practice_synergy'] = df['hours_studied'] * df['practice_tests']
df_engineered['age_experience_ratio'] = df['age'] / (df['years_experience'] + 1)  # +1 to avoid division by zero
df_engineered['income_per_year_exp'] = df['income'] / (df['years_experience'] + 1)

print("\n✓ Created interaction features:")
print("  • study_practice_synergy = hours_studied × practice_tests")
print("  • age_experience_ratio = age ÷ years_experience")
print("  • income_per_year_exp = income ÷ years_experience")

print("\n💡 Why interactions matter:")
print("   The COMBINATION of features often matters more than individual features!")

## Step 5: Feature Engineering Technique #2 - Polynomial Features

**Polynomial features** capture non-linear relationships.

Example: Motivation score of 8 might be MORE than twice as good as score of 4.

In [ ]:
print("\n" + "="*70)
print("FEATURE ENGINEERING #2: POLYNOMIAL FEATURES")
print("="*70)

# Create polynomial features
df_engineered['motivation_squared'] = df['motivation_score'] ** 2
df_engineered['hours_studied_squared'] = df['hours_studied'] ** 2
df_engineered['age_squared'] = df['age'] ** 2

print("\n✓ Created polynomial features:")
print("  • motivation_squared = motivation_score²")
print("  • hours_studied_squared = hours_studied²")
print("  • age_squared = age²")

print("\n💡 Why polynomial features matter:")
print("   Many relationships are NON-LINEAR!")
print("   Going from 5→6 might have different impact than 1→2")

## Step 6: Feature Engineering Technique #3 - Binning

**Binning** converts continuous variables into categories.

In [ ]:
print("\n" + "="*70)
print("FEATURE ENGINEERING #3: BINNING")
print("="*70)

# Create age groups
df_engineered['age_group'] = pd.cut(
    df['age'],
    bins=[0, 30, 50, 100],
    labels=['young', 'middle', 'senior']
)

# Create income brackets
df_engineered['income_bracket'] = pd.cut(
    df['income'],
    bins=[0, 50000, 100000, 200000],
    labels=['low', 'medium', 'high']
)

# Convert to numeric (label encoding for simplicity)
df_engineered['age_group_encoded'] = df_engineered['age_group'].cat.codes
df_engineered['income_bracket_encoded'] = df_engineered['income_bracket'].cat.codes

# Drop the categorical columns
df_engineered = df_engineered.drop(['age_group', 'income_bracket'], axis=1)

print("\n✓ Created binned features:")
print("  • age_group: young (18-30), middle (30-50), senior (50+)")
print("  • income_bracket: low (<50k), medium (50-100k), high (100k+)")

print("\n💡 Why binning matters:")
print("   Sometimes CATEGORIES work better than exact numbers!")

In [ ]:
# Show engineered features
print("\nDataset After Feature Engineering:")
print(f"Original features: {X_baseline.shape[1]}")
print(f"Engineered features: {df_engineered.shape[1] - 1}")
print(f"New features created: {df_engineered.shape[1] - 1 - X_baseline.shape[1]}")

print("\nAll features:")
print(list(df_engineered.columns))

## Step 7: Feature Scaling

Different features have different scales. This can hurt model performance!

**StandardScaler:** Mean=0, StdDev=1 (good for most algorithms)  
**MinMaxScaler:** Scale to [0, 1] range

In [ ]:
print("\n" + "="*70)
print("FEATURE ENGINEERING #4: FEATURE SCALING")
print("="*70)

# Prepare engineered features
X_engineered = df_engineered.drop('success', axis=1)

# Split data
X_train_eng, X_test_eng, y_train_eng, y_test_eng = train_test_split(
    X_engineered, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nBefore scaling:")
print("Feature ranges:")
for col in X_train_eng.columns[:5]:  # Show first 5
    print(f"  {col}: {X_train_eng[col].min():.1f} to {X_train_eng[col].max():.1f}")

# Apply StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_eng)
X_test_scaled = scaler.transform(X_test_eng)

# Convert back to DataFrame for clarity
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train_eng.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test_eng.columns)

print("\n✓ Scaling applied!")
print("\nAfter scaling:")
print("Feature ranges (should be roughly -3 to +3):")
for col in X_train_scaled.columns[:5]:  # Show first 5
    print(f"  {col}: {X_train_scaled[col].min():.2f} to {X_train_scaled[col].max():.2f}")

print("\n💡 Why scaling matters:")
print("   Algorithms like SVM and Logistic Regression are sensitive to scale!")
print("   Without scaling, 'income' (0-150,000) dominates 'motivation' (1-10)")

In [ ]:
# Visualize the effect of scaling
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Before scaling
X_train_eng[['age', 'income', 'motivation_score']].boxplot(ax=axes[0])
axes[0].set_title('Before Scaling', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Value', fontsize=12)
axes[0].grid(alpha=0.3)

# After scaling
X_train_scaled[['age', 'income', 'motivation_score']].boxplot(ax=axes[1])
axes[1].set_title('After Scaling', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Standardized Value', fontsize=12)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Now all features are on the same scale!")

## Step 8: Train Model with Engineered Features

In [ ]:
print("\n" + "="*70)
print("MODEL WITH FEATURE ENGINEERING")
print("="*70)

# Train Logistic Regression with engineered features
print("\nTraining Logistic Regression with engineered features...")
engineered_model = LogisticRegression(max_iter=1000, random_state=42)
engineered_model.fit(X_train_scaled, y_train)

# Evaluate
engineered_predictions = engineered_model.predict(X_test_scaled)
engineered_accuracy = accuracy_score(y_test, engineered_predictions)

print("\n" + "="*70)
print(f"ENGINEERED MODEL ACCURACY: {engineered_accuracy*100:.2f}%")
print("="*70)

# Calculate improvement
improvement = (engineered_accuracy - baseline_accuracy) * 100
improvement_pct = (improvement / baseline_accuracy) / 100 * 100

print("\n📊 COMPARISON:")
print(f"  Baseline (raw features):     {baseline_accuracy*100:.2f}%")
print(f"  Engineered (new features):   {engineered_accuracy*100:.2f}%")
print(f"  Improvement:                 +{improvement:.2f} percentage points")
print(f"  Relative improvement:        +{improvement_pct:.1f}%")

if improvement > 5:
    print("\n🎉 SIGNIFICANT IMPROVEMENT through feature engineering!")
elif improvement > 2:
    print("\n✓ Notable improvement through feature engineering!")
else:
    print("\n⚠️ Modest improvement - features may need more tuning.")

In [ ]:
# Detailed report
print("Engineered Model Classification Report:")
print(classification_report(y_test, engineered_predictions, target_names=['Fail', 'Success']))

## Step 9: Compare Multiple Models (Baseline vs Engineered)

In [ ]:
print("\n" + "="*70)
print("COMPARING MULTIPLE ALGORITHMS: BASELINE VS ENGINEERED")
print("="*70)

# Define algorithms
algorithms = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    'SVM': SVC(kernel='rbf', random_state=42)
}

results = []

for name, model in algorithms.items():
    print(f"\nTesting {name}...")
    
    # Baseline (raw features, unscaled)
    model_base = model.__class__(**model.get_params())
    model_base.fit(X_train_base, y_train)
    pred_base = model_base.predict(X_test_base)
    acc_base = accuracy_score(y_test, pred_base)
    
    # Engineered (new features, scaled)
    model_eng = model.__class__(**model.get_params())
    model_eng.fit(X_train_scaled, y_train)
    pred_eng = model_eng.predict(X_test_scaled)
    acc_eng = accuracy_score(y_test, pred_eng)
    
    improvement = (acc_eng - acc_base) * 100
    
    results.append({
        'Algorithm': name,
        'Baseline': acc_base,
        'Engineered': acc_eng,
        'Improvement': improvement
    })
    
    print(f"  Baseline:   {acc_base*100:.2f}%")
    print(f"  Engineered: {acc_eng*100:.2f}%")
    print(f"  Improvement: {improvement:+.2f}pp")

# Create comparison DataFrame
results_df = pd.DataFrame(results)

print("\n" + "="*70)
print("SUMMARY:")
display(results_df)

In [ ]:
# Visualize comparison
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(results_df))
width = 0.35

bars1 = ax.bar(x - width/2, results_df['Baseline']*100, width, 
               label='Baseline', color='#FF6B6B', alpha=0.8, edgecolor='black')
bars2 = ax.bar(x + width/2, results_df['Engineered']*100, width,
               label='Engineered', color='#4ECDC4', alpha=0.8, edgecolor='black')

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.1f}%',
                ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_xlabel('Algorithm', fontsize=12, fontweight='bold')
ax.set_ylabel('Accuracy (%)', fontsize=12, fontweight='bold')
ax.set_title('Impact of Feature Engineering on Different Algorithms', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(results_df['Algorithm'])
ax.legend(fontsize=11)
ax.set_ylim([50, 100])
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Feature engineering improved ALL algorithms!")

## Step 10: Feature Importance Analysis

In [ ]:
# Train Random Forest to get feature importance
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_model.fit(X_train_scaled, y_train)

# Get feature importance
feature_importance = pd.DataFrame({
    'Feature': X_train_scaled.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("Feature Importance (Top 10):")
print("="*60)
display(feature_importance.head(10))

# Visualize
plt.figure(figsize=(12, 6))
top_features = feature_importance.head(10)
plt.barh(top_features['Feature'], top_features['Importance'],
         color='#45B7D1', alpha=0.8, edgecolor='black')
plt.xlabel('Importance', fontsize=12, fontweight='bold')
plt.ylabel('Feature', fontsize=12, fontweight='bold')
plt.title('Top 10 Most Important Features', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

# Identify engineered features in top 10
engineered_features = [
    'study_practice_synergy', 'age_experience_ratio', 'income_per_year_exp',
    'motivation_squared', 'hours_studied_squared', 'age_squared',
    'age_group_encoded', 'income_bracket_encoded'
]

top_10_features = feature_importance.head(10)['Feature'].tolist()
engineered_in_top_10 = [f for f in top_10_features if f in engineered_features]

print(f"\n📊 {len(engineered_in_top_10)} out of top 10 features are ENGINEERED!")
print(f"   Engineered features in top 10: {engineered_in_top_10}")
print("\n✓ This proves that feature engineering creates valuable predictors!")

## Step 11: Summary and Key Takeaways

In [ ]:
print("\n" + "="*70)
print("PROJECT SUMMARY - FEATURE ENGINEERING FOR CLASSIFICATION")
print("="*70)

print("\n✅ WHAT YOU ACCOMPLISHED:")
print("  1. Created a dataset designed for feature engineering")
print("  2. Established baseline model performance")
print("  3. Created interaction features (A × B)")
print("  4. Created polynomial features (A²)")
print("  5. Binned continuous variables into categories")
print("  6. Applied feature scaling (StandardScaler)")
print(f"  7. Improved accuracy by {improvement:.2f} percentage points")
print("  8. Compared impact across multiple algorithms")
print("  9. Analyzed feature importance")

print("\n🎯 KEY LEARNINGS:")
print("  • Feature engineering can dramatically improve model performance")
print("  • Interaction features capture synergies between variables")
print("  • Polynomial features handle non-linear relationships")
print("  • Feature scaling is CRITICAL for many algorithms")
print("  • Different algorithms benefit differently from engineering")
print("  • Engineered features often become the most important!")

print("\n📊 FINAL RESULTS:")
print(f"  • Dataset: {len(df)} samples")
print(f"  • Original features: {X_baseline.shape[1]}")
print(f"  • Engineered features: {X_engineered.shape[1]}")
print(f"  • New features created: {X_engineered.shape[1] - X_baseline.shape[1]}")
print(f"  • Baseline accuracy: {baseline_accuracy*100:.2f}%")
print(f"  • Best engineered accuracy: {results_df['Engineered'].max()*100:.2f}%")
print(f"  • Maximum improvement: {results_df['Improvement'].max():.2f}pp")

print("\n🔧 FEATURE ENGINEERING TECHNIQUES USED:")
print("  1. Interaction Features:")
print("     • study_practice_synergy (hours × tests)")
print("     • age_experience_ratio (age ÷ experience)")
print("  2. Polynomial Features:")
print("     • motivation_squared")
print("     • hours_studied_squared")
print("  3. Binning:")
print("     • age_group (young/middle/senior)")
print("     • income_bracket (low/medium/high)")
print("  4. Scaling:")
print("     • StandardScaler (mean=0, std=1)")

print("\n🚀 NEXT STEPS:")
print("  • Session 6: Systematic model comparison")
print("  • Compare 5+ algorithms on same dataset")
print("  • Create ROC curves for all models")
print("  • Build a decision guide for algorithm selection")

print("\n💼 REAL-WORLD APPLICATIONS:")
print("  • Credit scoring (income × debt ratio)")
print("  • Marketing (customer engagement × purchase history)")
print("  • Healthcare (age × risk factors)")
print("  • E-commerce (views × time on site × cart value)")

print("\n" + "="*70)
print("🎉 YOU MASTERED FEATURE ENGINEERING!")
print("="*70)

print("\n💡 REMEMBER: Good features > Complex models!")
print("   Spend time on feature engineering - it's worth it!")

---

## AI Prompts for This Notebook

### Prompt 1: Dataset Creation
```
Create a synthetic binary classification dataset with 1000 samples where the target 
depends on interaction features (hours_studied * practice_tests) and polynomial 
features (motivation_score squared). Include features with different scales (age: 
18-80, income: 20k-150k) to demonstrate the importance of scaling.
```

### Prompt 2: Interaction Features
```
Show how to create interaction features from existing columns. Create features like 
'study_practice_synergy' (hours × tests) and 'income_per_year_exp' (income ÷ experience). 
Explain why interactions capture relationships that individual features miss.
```

### Prompt 3: Polynomial Features
```
Create polynomial features (squared terms) for key variables like motivation_score, 
hours_studied, and age. Explain when and why polynomial features help capture 
non-linear relationships in the data.
```

### Prompt 4: Feature Scaling
```
Apply StandardScaler to normalize all features. Show before/after boxplots comparing 
the scales of features like age (18-80) vs income (20k-150k) vs motivation (1-10). 
Explain why scaling matters for algorithms like SVM and Logistic Regression.
```

### Prompt 5: Before/After Comparison
```
Compare model performance before and after feature engineering. Train Logistic Regression, 
Random Forest, and SVM on both raw features and engineered features. Create a grouped 
bar chart showing the accuracy improvement for each algorithm.
```

### Prompt 6: Feature Importance
```
Use Random Forest to extract feature importance scores. Identify which engineered 
features appear in the top 10 most important features. Visualize with a horizontal 
bar chart.
```

---

**End of Notebook**

**Created:** 2026-01-06  
**Course:** ML for Engineers - Module 2  
**Project:** Feature Engineering for Classification  
**Target:** 10%+ Improvement ✅